In [1]:
#Sheet to Template
import pandas as pd
import numpy as np

In [2]:
# =========================================================
# READ SOURCE DATA
# =========================================================
pvtpo_df = pd.read_excel("Pvt PO draft.xlsx")
pvtfinal_df = pd.read_excel("Pvt Final draft.xlsx")


In [3]:
import pandas as pd

# --- Example date ---
order_date = pd.to_datetime("2026-08-30")

# --- Select required columns from pvtfinal_df ---
result_df = pvtfinal_df[[
    "PN AL78",
    "Long PN",
    "Description"
]].copy()

# --- Rename columns ---
result_df.rename(columns={
    "PN AL78": "Print PN#",
    "Long PN": "GOMS PN#"
}, inplace=True)

# --- Add fixed-value columns ---
result_df.insert(0, "Date", order_date)
result_df.insert(1, "MRC Location", "Indonesia")
result_df["MRC Ordered with (PDC Name)"] = "Singapore PDC"

# --- Add blank Supplier ID column ---
result_df["Supplier ID #"] = ""

# --- Format date like Excel ---
result_df["Date"] = result_df["Date"].dt.strftime("%d-%b-%y")

# --- Final column order ---
result_df = result_df[[
    "Date",
    "MRC Location",
    "Print PN#",
    "GOMS PN#",
    "Description",
    "MRC Ordered with (PDC Name)",
    "Supplier ID #"
]]

In [4]:
#PO NO
# --- Clean keys ---
lookup_df = pvtpo_df[["PN AL78", "PO"]].copy()
lookup_df["PN AL78"] = lookup_df["PN AL78"].astype(str).str.strip()

# --- Keep ONE PO per PN (Excel VLOOKUP behavior = first match) ---
lookup_df = lookup_df.drop_duplicates(subset=["PN AL78"])

# --- Build lookup dict ---
po_dict = lookup_df.set_index("PN AL78")["PO"].to_dict()
# --- Ensure key type matches ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

# --- VLOOKUP equivalent ---
result_df["MRC Notes - Order #"] = result_df["Print PN#"].map(po_dict)
result_df = result_df[[
    "Date",
    "MRC Location",
    "Print PN#",
    "GOMS PN#",
    "Description",
    "MRC Ordered with (PDC Name)",
    "Supplier ID #",
    "MRC Notes - Order #"
]]
# --- Add blank columns ---
result_df["Category"] = ""
result_df["Consideration"] = ""


In [5]:
# --- Build critical lookup ---
critical_lookup = (
    pvtfinal_df[["PN AL78", "critical"]]
    .copy()
)

critical_lookup["PN AL78"] = (
    critical_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Ensure one row per PN (safety) ---
critical_lookup = critical_lookup.drop_duplicates(subset=["PN AL78"])

# --- Convert to dict (Excel VLOOKUP behavior) ---
critical_dict = critical_lookup.set_index("PN AL78")["critical"].to_dict()
# --- Ensure key matches ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

# --- Add critical column ---
result_df["critical"] = result_df["Print PN#"].map(critical_dict)
result_df = result_df[[
    "Date",
    "MRC Location",
    "Print PN#",
    "GOMS PN#",
    "Description",
    "MRC Ordered with (PDC Name)",
    "Supplier ID #",
    "MRC Notes - Order #",
    "Category",
    "Consideration",
    "critical"
]]


In [6]:
#Engine
# --- Build Engine lookup ---
engine_lookup = (
    pvtfinal_df[["PN AL78", "Engine Model"]]
    .copy()
)

engine_lookup["PN AL78"] = (
    engine_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Safety: ensure one row per PN ---
engine_lookup = engine_lookup.drop_duplicates(subset=["PN AL78"])

# --- Convert to dict (Excel VLOOKUP behavior) ---
engine_dict = engine_lookup.set_index("PN AL78")["Engine Model"].to_dict()
# --- Ensure key matches ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

# --- Add Engine column ---
result_df["Engine"] = result_df["Print PN#"].map(engine_dict)
# --- Add blank columns ---
result_df["Engine Family"] = ""
result_df["Planning Code"] = ""



In [7]:
# --- Build Alloc On Hand Net lookup ---
alloc_lookup = (
    pvtfinal_df[["PN AL78", "Avrg.OH CM"]]
    .copy()
)

alloc_lookup["PN AL78"] = (
    alloc_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Safety: ensure one row per PN ---
alloc_lookup = alloc_lookup.drop_duplicates(subset=["PN AL78"])

# --- Convert to dict (Excel VLOOKUP behavior) ---
alloc_dict = alloc_lookup.set_index("PN AL78")["Avrg.OH CM"].to_dict()
# --- Ensure key matches ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

# --- Add Alloc On Hand Net column ---
result_df["Alloc On Hand Net"] = result_df["Print PN#"].map(alloc_dict)



In [8]:
# --- Build In Receiving lookup ---
inrcv_lookup = (
    pvtfinal_df[["PN AL78", "Avrg. InRcv"]]
    .copy()
)

inrcv_lookup["PN AL78"] = (
    inrcv_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Safety: ensure one row per PN ---
inrcv_lookup = inrcv_lookup.drop_duplicates(subset=["PN AL78"])

# --- Convert to dict (Excel VLOOKUP behavior) ---
inrcv_dict = inrcv_lookup.set_index("PN AL78")["Avrg. InRcv"].to_dict()
# --- Ensure key matches ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

# --- Add In receiving column ---
result_df["In receiving"] = result_df["Print PN#"].map(inrcv_dict)


In [ ]:
print(result_df)

In [9]:
LOOKUP_OUTPUT_COL = "8-2026"   # ← CURRENT MONTH
OUTPUT_COL_NAME = (
    "Receipt to MRC Under Allocation Reserved Pick (Current Month)"
)
def vlookup_map(result_df, source_df, key_left, key_right, value_col):
    lookup = (
        source_df[[key_right, value_col]]
        .copy()
    )

    lookup[key_right] = (
        lookup[key_right]
        .astype(str)
        .str.strip()
    )

    lookup = lookup.drop_duplicates(subset=[key_right])

    lookup_dict = lookup.set_index(key_right)[value_col].to_dict()

    result_df[key_left] = result_df[key_left].astype(str).str.strip()

    return result_df[key_left].map(lookup_dict)
result_df[OUTPUT_COL_NAME] = vlookup_map(
    result_df=result_df,
    source_df=pvtfinal_df,
    key_left="Print PN#",
    key_right="PN AL78",
    value_col=LOOKUP_OUTPUT_COL
).fillna("")
################################
LOOKUP_OUTPUT_COL_PLUS1 = "9-2026" # <----- NEXT MONTH
OUTPUT_COL_NAME_PLUS1 = (
    "Receipt to MRC Under Allocation Reserved Pick (+1 Month)"
)
result_df[OUTPUT_COL_NAME_PLUS1] = vlookup_map(
    result_df=result_df,
    source_df=pvtfinal_df,
    key_left="Print PN#",
    key_right="PN AL78",
    value_col=LOOKUP_OUTPUT_COL_PLUS1
).fillna("")
##################################
LOOKUP_OUTPUT_COL_PLUS2 = "10-2026" #<----- NEXT 2 MONTHS
OUTPUT_COL_NAME_PLUS2 = (
    "Receipt to MRC Under Allocation Reserved Pick (+2 Month)"
)
result_df[OUTPUT_COL_NAME_PLUS2] = vlookup_map(
    result_df=result_df,
    source_df=pvtfinal_df,
    key_left="Print PN#",
    key_right="PN AL78",
    value_col=LOOKUP_OUTPUT_COL_PLUS2
).fillna("")

In [10]:
# --- Build fixed lookup ---
intr_lookup = (
    pvtfinal_df[["PN AL78", "Avrg. InTr"]]
    .copy()
)

intr_lookup["PN AL78"] = (
    intr_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
intr_lookup = intr_lookup.drop_duplicates(subset=["PN AL78"])

intr_dict = intr_lookup.set_index("PN AL78")["Avrg. InTr"].to_dict()

# --- Apply lookup ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["In Transit Current Month"] = (
    result_df["Print PN#"].map(intr_dict).fillna("")
)
# --- Add blank columns ---
result_df["In Transit (+1 Month)"] = ""
result_df["In Transit (+2 Months)"] = ""


In [11]:
# --- Build fixed lookup for Open Orders Prior Months ---
open_orders_lookup = (
    pvtfinal_df[["PN AL78", "Sum 7 Months Prior"]]
    .copy()
)

open_orders_lookup["PN AL78"] = (
    open_orders_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
open_orders_lookup = open_orders_lookup.drop_duplicates(subset=["PN AL78"])

open_orders_dict = (
    open_orders_lookup
    .set_index("PN AL78")["Sum 7 Months Prior"]
    .to_dict()
)

# --- Apply lookup ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["Open Orders Prior Months"] = (
    result_df["Print PN#"].map(open_orders_dict).fillna("")
)
# --- Build fixed lookup for Open Orders (Current Month) ---
oo_cm_lookup = (
    pvtfinal_df[["PN AL78", "Sum OO CM"]]
    .copy()
)

oo_cm_lookup["PN AL78"] = (
    oo_cm_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
oo_cm_lookup = oo_cm_lookup.drop_duplicates(subset=["PN AL78"])

oo_cm_dict = (
    oo_cm_lookup
    .set_index("PN AL78")["Sum OO CM"]
    .to_dict()
)

# --- Apply lookup ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["Open Orders (Current Month)"] = (
    result_df["Print PN#"].map(oo_cm_dict).fillna("")
)
# --- Build fixed lookup for Open Orders (+1 Month) ---
oo_nm_lookup = (
    pvtfinal_df[["PN AL78", "Sum OO NM"]]
    .copy()
)

oo_nm_lookup["PN AL78"] = (
    oo_nm_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
oo_nm_lookup = oo_nm_lookup.drop_duplicates(subset=["PN AL78"])

oo_nm_dict = (
    oo_nm_lookup
    .set_index("PN AL78")["Sum OO NM"]
    .to_dict()
)

# --- Apply lookup ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["Open Orders (+1 Month)"] = (
    result_df["Print PN#"].map(oo_nm_dict).fillna("")
)
# --- Build fixed lookup for Open Orders (+2 Months) ---
oo_n2m_lookup = (
    pvtfinal_df[["PN AL78", "Sum OO N2M"]]
    .copy()
)

oo_n2m_lookup["PN AL78"] = (
    oo_n2m_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
oo_n2m_lookup = oo_n2m_lookup.drop_duplicates(subset=["PN AL78"])

oo_n2m_dict = (
    oo_n2m_lookup
    .set_index("PN AL78")["Sum OO N2M"]
    .to_dict()
)

# --- Apply lookup ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["Open Orders (+2 Months)"] = (
    result_df["Print PN#"].map(oo_n2m_dict).fillna("")
)


In [12]:
# --- Build fixed lookup for Build Requirements (Current Month) ---
build_req_lookup = (
    pvtfinal_df[["PN AL78", "Max.Curr. Month"]]
    .copy()
)

build_req_lookup["PN AL78"] = (
    build_req_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: take first occurrence only ---
build_req_lookup = build_req_lookup.drop_duplicates(subset=["PN AL78"])

build_req_dict = (
    build_req_lookup
    .set_index("PN AL78")["Max.Curr. Month"]
    .to_dict()
)

# --- Apply lookup to result_df ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["Build Requirements (Current Month)"] = (
    result_df["Print PN#"].map(build_req_dict).fillna("")
)


In [13]:
# --- Build fixed lookup for Build Requirement (+1 Month) ---
build_req_p1_lookup = (
    pvtfinal_df[["PN AL78", "Max.Req. CM"]]
    .copy()
)

build_req_p1_lookup["PN AL78"] = (
    build_req_p1_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
build_req_p1_lookup = build_req_p1_lookup.drop_duplicates(subset=["PN AL78"])

build_req_p1_dict = (
    build_req_p1_lookup
    .set_index("PN AL78")["Max.Req. CM"]
    .to_dict()
)

# --- Apply lookup ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()

result_df["Build Requirement (+1 Month)"] = (
    result_df["Print PN#"].map(build_req_p1_dict).fillna("")
)


In [14]:
# --- Build lookup for Build Requirements (+2 Months) ---
build_req_p2_lookup = (
    pvtfinal_df[["PN AL78", "Max.Req. NM"]]
    .copy()
)

build_req_p2_lookup["PN AL78"] = (
    build_req_p2_lookup["PN AL78"]
    .astype(str)
    .str.strip()
)

# --- Excel VLOOKUP behavior: first match only ---
build_req_p2_lookup = build_req_p2_lookup.drop_duplicates(subset=["PN AL78"])

build_req_p2_dict = (
    build_req_p2_lookup
    .set_index("PN AL78")["Max.Req. NM"]
    .to_dict()
)

# --- Apply lookup ---
result_df["Build Requirements (+2 Months)"] = (
    result_df["Print PN#"]
    .astype(str)
    .str.strip()
    .map(build_req_p2_dict)
    .fillna("")
)


In [15]:
# --- Ensure numeric columns (Excel-like behavior: blanks → 0) ---
calc1_cols = [
    "Alloc On Hand Net",
    "In receiving",
    "Receipt to MRC Under Allocation Reserved Pick (Current Month)",
    "In Transit Current Month",
    "Build Requirements (Current Month)"
]

for col in calc1_cols:
    result_df[col] = pd.to_numeric(result_df[col], errors="coerce").fillna(0)

# --- Calculate Shortages (Current Month) ---
result_df["Shortages (Current Month)"] = (
    result_df["Alloc On Hand Net"]
    + result_df["In receiving"]
    + result_df["Receipt to MRC Under Allocation Reserved Pick (Current Month)"]
    + result_df["In Transit Current Month"]
    - result_df["Build Requirements (Current Month)"]
)


In [16]:
# --- Ensure numeric columns (Excel-like behavior: blanks → 0) ---
calc2_cols = [
    "Receipt to MRC Under Allocation Reserved Pick (+1 Month)",
    "Shortages (Current Month)",
    "In Transit (+1 Month)",
    "Build Requirement (+1 Month)"
]

for col in calc2_cols:
    result_df[col] = pd.to_numeric(result_df[col], errors="coerce").fillna(0)

# --- Calculate Shortages (Current Month) ---
result_df["Shortages (+1 Month)"] = (
    result_df["Shortages (Current Month)"]
    + result_df["In Transit (+1 Month)"]
    + result_df["Receipt to MRC Under Allocation Reserved Pick (+1 Month)"]
    - result_df["Build Requirement (+1 Month)"]
)


In [17]:
# --- Ensure numeric columns (Excel-like behavior: blanks → 0) ---
calc3_cols = [
    "Receipt to MRC Under Allocation Reserved Pick (+2 Month)",
    "Shortages (+1 Month)",
    "In Transit (+2 Months)",
    "Build Requirements (+2 Months)"
]

for col in calc3_cols:
    result_df[col] = pd.to_numeric(result_df[col], errors="coerce").fillna(0)

# --- Calculate Shortages (Current Month) ---
result_df["Shortages (+2 Month)"] = (
    result_df["Shortages (+1 Month)"]
    + result_df["In Transit (+2 Months)"]
    + result_df["Receipt to MRC Under Allocation Reserved Pick (+2 Month)"]
    - result_df["Build Requirements (+2 Months)"]
)


In [18]:
# --- Add 3 blank spacer columns ---
result_df[""] = ""
result_df["  "] = ""
result_df["   "] = ""

# --- Columns from pvtfinal_df you want to append ---
cols_to_copy = [
    "PN No suffix",
    "8-2026",
    "9-2026",
    "10-2026",
    "11-2026",
    "12-2026",
    "Missing",
    "Grand Total",
    "chk avail.1",
    "Balnc.",
    "chk Resvrd",
    "chk avail.2",
    "add",
    "DN Prc",
    "OH AL78",
    "Cross to OH AL78",
    "critical",
    "Fin Business Code",
    "Engine Model"
]

# --- Clean join keys ---
result_df["Print PN#"] = result_df["Print PN#"].astype(str).str.strip()
pvtfinal_df["PN AL78"] = pvtfinal_df["PN AL78"].astype(str).str.strip()

# --- Build lookup table ---
append_df = (
    pvtfinal_df[["PN AL78"] + cols_to_copy]
    .drop_duplicates(subset=["PN AL78"])
    .rename(columns={"critical": "critical_2"})  # 👈 rename here
)

# --- Merge to the RIGHT ---
result_df = result_df.merge(
    append_df,
    how="left",
    left_on="Print PN#",
    right_on="PN AL78"
)

# --- Remove helper column ---
result_df.drop(columns=["PN AL78"], inplace=True)


In [19]:
# --- Ensure chk avail. exists and is clean ---
result_df["chk avail.2"] = (
    result_df["chk avail.2"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# --- Keep only rows where chk avail. == 'check' ---
result_df = (
    result_df[result_df["chk avail.2"] == "check"]
    .reset_index(drop=True)
)


In [20]:
print(result_df)

          Date MRC Location     Print PN#     GOMS PN#  \
0    30-Aug-26    Indonesia        101843    010184300   
1    30-Aug-26    Indonesia        103023    010302300   
2    30-Aug-26    Indonesia        106221    010622100   
3    30-Aug-26    Indonesia        119363    011936300   
4    30-Aug-26    Indonesia        129839    012983900   
..         ...          ...           ...          ...   
321  30-Aug-26    Indonesia  S   167    A  S00016700 A   
322  30-Aug-26    Indonesia  S   188    C  S00018800 C   
323  30-Aug-26    Indonesia       S   285    S00028500   
324  30-Aug-26    Indonesia       S   622    S00062200   
325  30-Aug-26    Indonesia       S  2268    S00226800   

                Description MRC Ordered with (PDC Name) Supplier ID #  \
0                      SHIM               Singapore PDC                 
1    SCREW,HEXAGON HEAD CAP               Singapore PDC                 
2                      CLIP               Singapore PDC                 
3         G

In [21]:
# =========================================================
# EXPORT
# =========================================================
result_df.to_excel("to Template draft.xlsx", index=False)

In [22]:
import pandas as pd
from datetime import datetime

# --- Create date string like 06Dec2025 ---
today_str = datetime.today().strftime("%d%b%Y")

# --- List of input files ---
files_and_sheets = {
    "to Template draft.xlsx": "to Template",
    "Pvt Final draft.xlsx": "Pvt Final",
    "Pvt draft.xlsx": "Pvt",
    "Pvt PO draft.xlsx": "Pvt PO",
    "PTC Draft.xlsx": "PTC",
    "PTC Normal Draft.xlsx": "PTC Normal",
    "Pvt chkPN draft.xlsx": "Pvt chkPN",
    "Weekly PTA CGL RTR Revised Open Orders Report 30Aug2026.xlsx": "All"
}

# --- Output file with date ---
output_file = f"Open Order Report Edited {today_str} Edited.xlsx"

# --- Create combined Excel file ---
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    for file, sheet_name in files_and_sheets.items():
        df = pd.read_excel(file)
        df.to_excel(writer, sheet_name=sheet_name, index=False)
